<a href="https://colab.research.google.com/github/TViguini/agentes-2-equipe-01/blob/main/Aula_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "openai>=1.99.0,<3"

In [2]:
import os, time, json, re, unicodedata
from openai import OpenAI


def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor


LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_API_KEY = obter_chave("GROQ_API_KEY")

MODELOS_CANDIDATOS = ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]
LLM_MODEL = MODELOS_CANDIDATOS[0]

# Preco publicado em console.groq.com/pricing, conferido em 21/08/2026.
# Dolares por MILHAO de tokens.
PRECOS = {
    "openai/gpt-oss-20b":  {"entrada": 0.075, "saida": 0.30},
    "openai/gpt-oss-120b": {"entrada": 0.150, "saida": 0.60},
}
TPM = 8_000

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print("chave carregada, termina em:", LLM_API_KEY[-4:])
print("modelo:", LLM_MODEL)
print(f"TPM do plano gratuito: {TPM:,}")

chave carregada, termina em: Qe6G
modelo: openai/gpt-oss-20b
TPM do plano gratuito: 8,000


In [3]:
def _espera_sugerida(erro, padrao: float) -> float:
    """Le o cabecalho retry-after, se o provedor mandou. Senao usa o padrao."""
    try:
        cab = getattr(getattr(erro, "response", None), "headers", {}) or {}
        v = cab.get("retry-after") or cab.get("Retry-After")
        if v:
            return float(v)
    except (TypeError, ValueError):
        pass
    return padrao


def chamar(mensagens, temperatura: float = 0.0, tentativas: int = 5,
           max_tokens: int = 2000, **extra):
    """Chama o modelo tolerando 429/503, com espera crescente.

    IMPORTANTE: gpt-oss e modelo de raciocinio, e os tokens de raciocinio
    consomem o orcamento de SAIDA antes do texto final. Com max_tokens curto,
    o raciocinio ocupa tudo e a resposta volta cortada — sem erro.

    NAO troca de modelo depois de N falhas, ao contrario do Encontro 3. Hoje o
    modelo tem de ser constante: a variavel do experimento e o DESENHO das
    ferramentas, e uma troca silenciosa no meio da medicao a invalidaria.
    """
    espera = 2.0
    for t in range(tentativas):
        try:
            r = cliente.chat.completions.create(
                model=LLM_MODEL, messages=mensagens,
                temperature=temperatura, max_tokens=max_tokens, **extra)
            if r.choices[0].finish_reason == "length":
                print("  [AVISO: resposta CORTADA por max_tokens. Aumente max_tokens.]")
            return r
        except Exception as e:
            codigo = getattr(e, "status_code", None)
            if codigo not in (429, 500, 502, 503, 504) or t == tentativas - 1:
                raise
            pausa = _espera_sugerida(e, espera)
            print(f"  [{codigo}] esperando {pausa:.0f}s  (nao invalida a medicao)")
            time.sleep(pausa)
            espera *= 2
    raise RuntimeError("todas as tentativas falharam")


def custo_usd(entrada: int, saida: int, modelo: str = None) -> float:
    """Custo em dolares, pelo preco publicado do Groq."""
    p = PRECOS[modelo or LLM_MODEL]
    return (entrada * p["entrada"] + saida * p["saida"]) / 1_000_000


print("chamada e calculo de custo prontos.")

chamada e calculo de custo prontos.


In [6]:
def temperatura_camara(camara: str) -> str:
    """Le a temperatura atual de uma camara fria e devolve o valor em graus Celsius."""
    leituras = {"CF-01": 4.2, "CF-02": 9.8, "CF-03": -21.5}
    if camara not in leituras:
        return f"camara {camara} desconhecida"
    return f"{camara}: {leituras[camara]} graus Celsius neste momento"


def especificacao_do_insumo(lote: str) -> str:
    """Devolve em que camara um lote esta guardado, a faixa permitida e a validade."""
    fichas = {
        "L-77": ("reagente enzimatico; guardado na camara CF-02; "
                 "faixa permitida de 2 a 8 C; validade 2026-11-30"),
        "L-88": ("meio de cultura; guardado na camara CF-01; "
                 "faixa permitida de 2 a 8 C; validade 2026-09-15"),
        "L-91": ("enzima de restricao; guardado na camara CF-03; "
                 "faixa permitida de -25 a -15 C; validade 2027-02-28"),
    }
    if lote not in fichas:
        return f"sem especificacao para o lote {lote}"
    # A saida DIZ de qual lote ela fala. Com um lote isso e redundante; com
    # dois, e o que torna o rastro legivel — ver a nota da Parte 2.
    return f"{lote}: {fichas[lote]}"


def historico_excursoes(camara: str, horas: str = "24") -> str:
    """Lista as excursoes de temperatura de uma camara nas ultimas N horas."""
    base = {
        "CF-01": [],
        "CF-02": [("-3h", "subiu a 9,8 C e ainda nao voltou"),
                  ("-19h", "pico de 8,6 C por cerca de 40 min")],
        "CF-03": [("-30h", "queda a -28 C por cerca de 15 min")],
    }
    if camara not in base:
        return f"camara {camara} desconhecida"
    try:
        janela = int(float(horas))
    except (TypeError, ValueError):
        return "ERRO: horas deve ser um numero, por exemplo 24"
    dentro = [f"{q} {d}" for q, d in base[camara] if int(q.strip("-h")) <= janela]
    if not dentro:
        return f"{camara}: 0 excursoes nas ultimas {janela}h"
    return (f"{camara}: {len(dentro)} excursao(oes) nas ultimas {janela}h -- "
            + "; ".join(dentro))


print("as tres funcoes do Encontro 3, com UMA linha alterada:")
print("  especificacao_do_insumo agora PREFIXA a saida com o lote.")
print()
print(especificacao_do_insumo("L-77"))
print(especificacao_do_insumo("L-88"))
print()
print("Sem esse prefixo, as duas linhas acima sairiam quase identicas no")
print("rastro — e ninguem, nem o modelo nem voce, saberia qual e qual.")

as tres funcoes do Encontro 3, com UMA linha alterada:
  especificacao_do_insumo agora PREFIXA a saida com o lote.

L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
L-88: meio de cultura; guardado na camara CF-01; faixa permitida de 2 a 8 C; validade 2026-09-15

Sem esse prefixo, as duas linhas acima sairiam quase identicas no
rastro — e ninguem, nem o modelo nem voce, saberia qual e qual.


In [7]:
# ---------------------------------------------------------------------
# As FERRAMENTAS COMPOSTAS. Note que elas nao acessam dado nenhum novo:
# so chamam as tres funcoes originais numa ordem fixa — decidida por MIM,
# em Python, e nao pelo modelo em tempo de execucao.
# ---------------------------------------------------------------------
PADRAO_CAMARA = re.compile(r"CF-\d\d")


def ficha_e_leitura(lote: str) -> str:
    """Devolve a ficha do LOTE e a temperatura ATUAL da camara onde ele esta.

    Junta duas consultas numa: a ficha (camara, faixa permitida, validade) e a
    leitura da camara indicada nela. Nao devolve historico: para saber se a
    camara saiu da faixa antes, use historico_excursoes.

    Args:
        lote: identificador do lote, no formato L-NN. Exemplo: "L-77"
    """
    ficha = especificacao_do_insumo(lote)
    m = PADRAO_CAMARA.search(ficha)
    if not m:
        return ficha
    return f"{ficha}\n{temperatura_camara(m.group(0))}"


def avaliar_lote(lote: str) -> str:
    """Devolve o laudo completo de um LOTE: a ficha, a temperatura ATUAL da
    camara onde ele esta, e as excursoes das ultimas 24 horas.

    Args:
        lote: identificador do lote, no formato L-NN. Exemplo: "L-77"
    """
    ficha = especificacao_do_insumo(lote)
    m = PADRAO_CAMARA.search(ficha)
    if not m:
        return ficha
    camara = m.group(0)                      # <<< QUEM DECIDIU ISSO? O Python.
    leitura = temperatura_camara(camara)
    historico = historico_excursoes(camara, "24")   # <<< E ISSO? Tambem.
    return f"{ficha}\n{leitura}\n{historico}"


print("ferramentas compostas prontas.")
print()
print("Compare os dois rastros que o agente vai produzir:")
print("  desenho fino  : especificacao_do_insumo -> temperatura_camara -> historico_excursoes")
print("  desenho grosso: avaliar_lote")
print()
print("Os dois leem os MESMOS dados. No segundo, a ORDEM foi decidida em Python.")

ferramentas compostas prontas.

Compare os dois rastros que o agente vai produzir:
  desenho fino  : especificacao_do_insumo -> temperatura_camara -> historico_excursoes
  desenho grosso: avaliar_lote

Os dois leem os MESMOS dados. No segundo, a ORDEM foi decidida em Python.


In [8]:
# ---------------------------------------------------------------------
# As tres declaracoes. Repare que as descriptions dizem ESCOPO — o que a
# ferramenta devolve e o que ela NAO cobre — e nunca ORDEM DE USO. Essa regra
# vem do Encontro 4: ordem de uso pertence a instrucao, senao as variaveis
# do experimento se misturam.
# ---------------------------------------------------------------------
def _param(nome, descricao, obrigatorio=True, extras=None):
    props = {nome: {"type": "string", "description": descricao}}
    if extras:
        props.update(extras)
    return {"type": "object", "properties": props,
            "required": [nome] if obrigatorio else []}


_D_LOTE = "identificador do lote, no formato L-NN. Exemplo: L-77"
_D_CAMARA = "identificador da camara, no formato CF-NN. Exemplo: CF-02"
_D_HORAS = {"horas": {"type": "string",
                      "description": "janela em horas, como texto. Exemplo: 24"}}


def _f(nome, descricao, parametros):
    return {"type": "function", "function": {
        "name": nome, "description": descricao, "parameters": parametros}}


TOOLS_FINA = [
    _f("especificacao_do_insumo",
       "Devolve a ficha de um LOTE: em que camara ele esta guardado, a faixa de "
       "temperatura permitida para ele, e a data de validade. Nao devolve leitura "
       "de temperatura: para isso use temperatura_camara.",
       _param("lote", _D_LOTE)),
    _f("temperatura_camara",
       "Le a temperatura ATUAL de uma camara fria e devolve o valor em graus "
       "Celsius. Nao devolve historico: para saber se a camara saiu da faixa "
       "antes, use historico_excursoes.",
       _param("camara", _D_CAMARA)),
    _f("historico_excursoes",
       "Lista as excursoes de temperatura de uma camara nas ultimas N horas. Uma "
       "excursao e um periodo em que a camara saiu da faixa permitida. Devolve "
       "historico, nao a condicao atual: para a leitura de agora use "
       "temperatura_camara.",
       _param("camara", _D_CAMARA, extras=_D_HORAS)),
]

TOOLS_MEDIA = [
    _f("ficha_e_leitura",
       "Devolve, de uma vez, a ficha de um LOTE (camara, faixa de temperatura "
       "permitida e validade) e a temperatura ATUAL da camara onde ele esta. Nao "
       "devolve historico: para isso use historico_excursoes.",
       _param("lote", _D_LOTE)),
    _f("historico_excursoes",
       "Lista as excursoes de temperatura de uma camara nas ultimas N horas. Uma "
       "excursao e um periodo em que a camara saiu da faixa permitida.",
       _param("camara", _D_CAMARA, extras=_D_HORAS)),
]

TOOLS_GROSSA = [
    _f("avaliar_lote",
       "Devolve o laudo completo de um LOTE: a ficha (camara, faixa de temperatura "
       "permitida, validade), a temperatura ATUAL da camara onde ele esta, e as "
       "excursoes de temperatura das ultimas 24 horas.",
       _param("lote", _D_LOTE)),
]

# O registro e a fronteira de seguranca, e ele acompanha o desenho: so um nome
# que esta no dicionario do desenho corrente pode virar chamada.
DESENHOS = {
    "fina": {
        "tools": TOOLS_FINA,
        "registro": {"especificacao_do_insumo": especificacao_do_insumo,
                     "temperatura_camara": temperatura_camara,
                     "historico_excursoes": historico_excursoes},
    },
    "media": {
        "tools": TOOLS_MEDIA,
        "registro": {"ficha_e_leitura": ficha_e_leitura,
                     "historico_excursoes": historico_excursoes},
    },
    "grossa": {
        "tools": TOOLS_GROSSA,
        "registro": {"avaliar_lote": avaliar_lote},
    },
}

# INVARIANTES. Se algum falhar, o experimento nao e atribuivel.
_PROIBIDO = ["sempre primeiro", "primeiro quando", "comece pela", "depois use"]
for _nome, _d in DESENHOS.items():
    _declaradas = {t["function"]["name"] for t in _d["tools"]}
    assert _declaradas == set(_d["registro"]), (
        f"desenho {_nome}: declaracao e registro divergem "
        f"-> {_declaradas ^ set(_d['registro'])}")
    for _t in _d["tools"]:
        _desc = _t["function"]["description"].lower()
        assert not any(p in _desc for p in _PROIBIDO), (
            f"a description de {_t['function']['name']} da ORDEM DE USO. "
            "Isso pertence a instrucao (regra do Encontro 4).")

print(f"{'desenho':>10}  {'ferramentas':>11}  {'caracteres da declaracao':>24}")
print("-" * 50)
for _nome, _d in DESENHOS.items():
    print(f"{_nome:>10}  {len(_d['tools']):>11}  {len(json.dumps(_d['tools'])):>24}")
print()
print("A ultima coluna e a PARCELA FIXA: ela vai no prompt de TODA volta.")
print("Guarde os tres numeros — a diferenca entre eles e metade da economia.")

   desenho  ferramentas  caracteres da declaracao
--------------------------------------------------
      fina            3                      1469
     media            2                       939
    grossa            1                       444

A ultima coluna e a PARCELA FIXA: ela vai no prompt de TODA volta.
Guarde os tres numeros — a diferenca entre eles e metade da economia.


In [9]:
INSTRUCAO = """Voce e um agente que avalia se insumos refrigerados podem ser usados.

Use as ferramentas disponiveis para obter os dados. Nunca invente uma leitura
nem uma faixa de temperatura: se precisa de um dado, chame a ferramenta.

Quando tiver todos os dados, responda em texto ao responsavel pelo almoxarifado.
Cite SEMPRE, para cada lote avaliado: a leitura em graus Celsius, a faixa
permitida do lote, e ha quanto tempo a camara esta fora da faixa (ou que nao
houve excursao). Sem esses numeros a resposta nao serve para auditoria.
"""


def executar(nome: str, argumentos_json: str, registro: dict) -> str:
    """Roda a ferramenta pedida. Devolve SEMPRE texto, nunca excecao.

    O registro e a fronteira de seguranca, e ele vem do DESENHO corrente:
    uma ferramenta que nao esta no desenho em teste nao pode ser chamada,
    mesmo que exista em Python.
    """
    if nome not in registro:
        return (f"ERRO: ferramenta '{nome}' nao existe neste agente. "
                f"Disponiveis: {', '.join(registro)}")
    try:
        args = json.loads(argumentos_json or "{}")
    except json.JSONDecodeError as e:
        return f"ERRO: os argumentos nao sao JSON valido: {e}"
    if not isinstance(args, dict):
        return "ERRO: os argumentos precisam ser um objeto JSON"
    try:
        return registro[nome](**args)
    except TypeError as e:
        return f"ERRO: argumentos invalidos para {nome}: {e}"


def mal_aproveitada(observacao: str) -> bool:
    """A chamada trouxe dado util, ou voltou vazia de informacao?"""
    o = (observacao or "").lower()
    return (o.startswith("erro") or "desconhecida" in o
            or "sem especificacao" in o)


def agente(pergunta: str, desenho: str = "fina", instrucao: str = None,
           max_iteracoes: int = 8, verboso: bool = False, **extra):
    """Roda o laco no canal nativo, com o DESENHO de ferramentas escolhido.

    Args:
        pergunta: o que o responsavel quer saber
        desenho: "fina", "media" ou "grossa"
        instrucao: por padrao a INSTRUCAO unica — a mesma para os tres desenhos
        max_iteracoes: teto de voltas. 8, nao 6: a pergunta de dois lotes no
            desenho fino pode precisar de mais, se o modelo nao paralelizar.
        extra: repassado a API
    """
    d = DESENHOS[desenho]
    tools, registro = d["tools"], d["registro"]
    mensagens = [{"role": "system", "content": instrucao or INSTRUCAO},
                 {"role": "user", "content": pergunta}]
    entrada = saida = chamadas = ruins = voltas_com_chamada = 0

    for volta in range(1, max_iteracoes + 1):
        r = chamar(mensagens, tools=tools, **extra)
        msg = r.choices[0].message
        entrada += r.usage.prompt_tokens
        saida += r.usage.completion_tokens

        if verboso:
            print(f"--- volta {volta} " + "-" * 46)

        if not msg.tool_calls:
            texto = msg.content or ""
            if verboso:
                _t = texto.strip()
                print("RESPOSTA FINAL:", _t[:1200])
                if len(_t) > 1200:
                    print(f"  [... +{len(_t) - 1200} caracteres. Corte do PRINT, "
                          f"nao da resposta.]")
            return {"resposta": texto, "desenho": desenho, "voltas": volta,
                    "voltas_com_chamada": voltas_com_chamada,
                    "chamadas": chamadas, "ruins": ruins,
                    "entrada": entrada, "saida": saida, "concluiu": True,
                    "modelo": LLM_MODEL}

        voltas_com_chamada += 1
        mensagens.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })

        if verboso and len(msg.tool_calls) > 1:
            print(f"  [{len(msg.tool_calls)} chamadas NESTA volta "
                  f"— e paralelismo, e economiza voltas]")

        for tc in msg.tool_calls:
            obs = executar(tc.function.name, tc.function.arguments, registro)
            chamadas += 1
            perdida = mal_aproveitada(obs)
            ruins += perdida
            if verboso:
                marca = "  [MAL APROVEITADA]" if perdida else ""
                print(f"  {tc.function.name}({tc.function.arguments}){marca}")
                print(f"    -> {obs}")
            mensagens.append({"role": "tool", "tool_call_id": tc.id,
                              "name": tc.function.name, "content": obs})

    return {"resposta": f"PAREI POR ORCAMENTO em {max_iteracoes} voltas.",
            "desenho": desenho, "voltas": max_iteracoes,
            "voltas_com_chamada": voltas_com_chamada, "chamadas": chamadas,
            "ruins": ruins, "entrada": entrada, "saida": saida,
            "concluiu": False, "modelo": LLM_MODEL}


def paralelismo(r: dict) -> float:
    """chamadas / voltas que fizeram chamada. 1.0 = totalmente sequencial."""
    return r["chamadas"] / r["voltas_com_chamada"] if r["voltas_com_chamada"] else 0.0


print("agente pronto, com os tres desenhos e a metrica de paralelismo.")
print(f"instrucao unica: {len(INSTRUCAO)} caracteres, a MESMA para os tres.")

agente pronto, com os tres desenhos e a metrica de paralelismo.
instrucao unica: 527 caracteres, a MESMA para os tres.


In [10]:
_SOSIAS = {ord(c): "-" for c in "‐‑‒–—―−﹘﹣－"}   # todos os tracos
_SOSIAS[0x00A0] = " "    # espaco inquebravel
_SOSIAS[0x202F] = " "    # espaco estreito inquebravel


def _normalizar(s: str) -> str:
    """Minusculas, sem acento, tracos e espacos sosias dobrados, virgula -> ponto."""
    s = unicodedata.normalize("NFD", (s or "").lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.translate(_SOSIAS).replace(",", ".")


UNIDADE = r"(?:\s*°?\s*(?:c|celsius|graus?)\b)?"
SEPARADOR = r"(?:a|ate|e|-|--|–|—|/|to)"

FATOS = {
    "leitura": [r"9\.8"],
    "faixa": [rf"\b2{UNIDADE}\s*{SEPARADOR}\s*8\b",
              r"entre\s+2\b[^\d\n]{0,14}?8\b",
              r"\b2\b[^\d\n]{0,14}?\b8\b"],
    "tempo": [r"\b3(?:\.0+)?\s*h(?:ora|our)?s?\b", r"tres\s+horas?\b",
              r"\b1[78]\d\s*min"],
}


def detalhar_evidencia(resposta: str) -> dict:
    t = _normalizar(resposta or "")
    return {n: any(re.search(p, t) for p in ps) for n, ps in FATOS.items()}


def avaliar_evidencia(resposta: str) -> int:
    """Nota de 0 a 3: quantos dos tres fatos do L-77 a resposta cita.

    NAO avalia se a resposta esta correta — avalia se ela e auditavel.
    """
    return sum(detalhar_evidencia(resposta).values())


def mencionou_todos(resposta: str, lotes) -> bool:
    """Todos os lotes pedidos aparecem na resposta?

    Existe porque a rubrica de evidencia foi feita para UM lote. Na pergunta de
    dois, uma resposta pode tirar 3/3 citando so o L-77 e ignorando o L-88 —
    e isso e uma falha grave que a evidencia sozinha nao pega.
    """
    t = _normalizar(resposta or "")
    return all(_normalizar(l) in t for l in lotes)


_pobres = [("O lote L-77 nao pode ser usado.", 0),
           ("Nao recomendo o uso; a temperatura esta fora do especificado.", 0),
           ("A camara marca 9,8 C e esta fora da faixa. Nao use.", 1),
           ("CF-02: 2 excursoes nas ultimas 24h. Pico de 8,6 C por 40 min.", 0),
           ("Faixa de 2 a 8 C. Nao ha leitura disponivel.", 1)]
_ricas = [
    "Nao use o L-77: a camara CF-02 marca 9,8 C contra a faixa permitida de "
    "2 a 8 C, e esta fora ha cerca de 3 horas.",
    "Leitura de 9.8 graus, faixa 2-8 graus, fora da faixa por 3h.",
    "A camara esta a 9,8 C, entre 2 e 8 C e o permitido, e ja sao 3,0 horas fora.",
    "9,8 C medidos; especificacao 2 C a 8 C; excursao iniciada ha 3 horas.",
    # a que quebrou o avaliador na execucao real de 24/08:
    "Faixa permitida: 2 °C a 8 °C. Atual: 9,8 °C. Fora ha aproximadamente 3 horas.",
    "Faixa 2°C–8°C, leitura 9,8°C, ~3h fora.",
    "Permitido de 2 ate 8 graus Celsius; medido 9,8; tempo fora 3 h.",
    "range 2 to 8 C, reading 9.8 C, out for 3 hours",
]
for _t, _e in _pobres:
    assert avaliar_evidencia(_t) == _e, f"FALSO POSITIVO em: {_t}"
for _t in _ricas:
    assert avaliar_evidencia(_t) == 3, f"FALSO NEGATIVO em: {_t}"
assert mencionou_todos("o L-77 e o L-88 estao ok", ["L-77", "L-88"])
assert not mencionou_todos("o L-77 esta fora da faixa", ["L-77", "L-88"])

print(f"avaliador conferido: {len(_pobres)} pobres e {len(_ricas)} ricas, "
      "todas no valor esperado.")
print("As ricas dizem o MESMO conteudo em formatos diferentes. Rubrica que")
print("pune estilo nao mede conteudo.")

avaliador conferido: 5 pobres e 8 ricas, todas no valor esperado.
As ricas dizem o MESMO conteudo em formatos diferentes. Rubrica que
pune estilo nao mede conteudo.


In [11]:
PERGUNTA_1 = "O lote L-77 ainda pode ser usado?"
PERGUNTA_2 = "Os lotes L-77 e L-88 ainda podem ser usados?"

PERGUNTAS = {
    "P1": {"texto": PERGUNTA_1, "lotes": ["L-77"], "n_padrao": 2},
    # n=1 por causa do TPM: uma execucao de dois lotes usa ~88% dos 8.000.
    "P2": {"texto": PERGUNTA_2, "lotes": ["L-77", "L-88"], "n_padrao": 1},
}

print("P1:", PERGUNTA_1, " -> cadeia de dependencia, paralelismo impossivel")
print("P2:", PERGUNTA_2, " -> dois pedidos independentes, paralelismo possivel")
print()
print("O L-88 esta na CF-01, a 4,2 C, DENTRO da faixa de 2 a 8, sem excursoes.")
print("Ou seja: a resposta certa para P2 e 'o L-77 nao, o L-88 sim'.")
print("Uma resposta que condena os dois esta ERRADA — e a evidencia nao pega")
print("isso. Repare nisso ao ler o rastro; e o assunto do Encontro 13.")

P1: O lote L-77 ainda pode ser usado?  -> cadeia de dependencia, paralelismo impossivel
P2: Os lotes L-77 e L-88 ainda podem ser usados?  -> dois pedidos independentes, paralelismo possivel

O L-88 esta na CF-01, a 4,2 C, DENTRO da faixa de 2 a 8, sem excursoes.
Ou seja: a resposta certa para P2 e 'o L-77 nao, o L-88 sim'.
Uma resposta que condena os dois esta ERRADA — e a evidencia nao pega
isso. Repare nisso ao ler o rastro; e o assunto do Encontro 13.


In [12]:
def medir(desenho: str, pergunta: str = "P1", n: int = None,
          verboso: bool = True) -> dict:
    """Roda o agente n vezes num par (desenho, pergunta) e resume.

    Args:
        desenho: "fina", "media" ou "grossa"
        pergunta: "P1" (um lote) ou "P2" (dois lotes)
        n: quantas execucoes. Por padrao 2 em P1 e 1 em P2 — em P2 uma execucao
           ja usa ~88% do TPM de 8.000, e duas estouram.
    """
    p = PERGUNTAS[pergunta]
    n = p["n_padrao"] if n is None else n
    modelo_no_inicio = LLM_MODEL

    notas, ent, sai = [], 0, 0
    voltas = chamadas = vcc = ruins = concluidas = cobriu = 0

    for i in range(1, n + 1):
        r = agente(p["texto"], desenho=desenho)
        nota = avaliar_evidencia(r["resposta"])
        todos = mencionou_todos(r["resposta"], p["lotes"])
        notas.append(nota)
        voltas += r["voltas"]; chamadas += r["chamadas"]
        vcc += r["voltas_com_chamada"]; ruins += r["ruins"]
        concluidas += r["concluiu"]; cobriu += todos
        ent += r["entrada"]; sai += r["saida"]
        if verboso:
            print(f"  exec {i}: {r['voltas']} voltas | {r['chamadas']} chamadas "
                  f"({paralelismo(r):.2f} por volta) | evidencia {nota}/3 | "
                  f"{'cobriu os lotes' if todos else 'FALTOU LOTE'} | "
                  f"{r['entrada']}+{r['saida']} tokens")

    assert LLM_MODEL == modelo_no_inicio, "o modelo mudou no meio da medicao"
    total = custo_usd(ent, sai)
    res = {
        "desenho": desenho, "pergunta": pergunta, "n": n, "modelo": LLM_MODEL,
        "voltas": voltas / n, "chamadas": chamadas / n,
        "paralelismo": (chamadas / vcc) if vcc else 0.0,
        "evidencia": sum(notas) / n, "cobriu": cobriu, "ruins": ruins,
        "entrada": ent / n, "saida": sai / n,
        "tokens": (ent + sai) / n,
        "custo_exec": total / n,
        "declaracao_car": len(json.dumps(DESENHOS[desenho]["tools"])),
    }
    if verboso:
        print(f"\n  [{desenho} · {pergunta}] modelo {LLM_MODEL}, n={n}")
        print(f"  voltas {res['voltas']:.1f} | chamadas {res['chamadas']:.1f} | "
              f"PARALELISMO {res['paralelismo']:.2f}")
        print(f"  evidencia {res['evidencia']:.2f}/3 | cobriu {cobriu}/{n} | "
              f"US$ {res['custo_exec']:.6f} por execucao")
        print(f"  tokens {res['tokens']:.0f} por execucao "
              f"({res['tokens'] / TPM * 100:.0f}% do TPM)")
    return res


print("funcao de medir pronta.")

funcao de medir pronta.


In [13]:
# CALIBRACAO. Uma execucao com rastro — e ela mede quanto custa no SEU caso.
print("=" * 66)
print("DESENHO FINO · um lote · com rastro")
print("=" * 66)
_demo = agente(PERGUNTA_2, desenho="fina", verboso=True)

_tk = _demo["entrada"] + _demo["saida"]
print()
print("-" * 66)
print(f"voltas            : {_demo['voltas']}  "
      f"({_demo['voltas_com_chamada']} fizeram chamada)")
print(f"chamadas          : {_demo['chamadas']}")
print(f"PARALELISMO       : {paralelismo(_demo):.2f}  "
      f"({'sequencial' if paralelismo(_demo) < 1.5 else 'paralelizou!'})")
print(f"evidencia         : {avaliar_evidencia(_demo['resposta'])}/3")
print(f"tokens            : {_demo['entrada']} + {_demo['saida']} = {_tk}")
print(f"custo             : US$ {custo_usd(_demo['entrada'], _demo['saida']):.6f}")
print()
print(f"Uma execucao usa {_tk / TPM * 100:.0f}% do seu TPM. "
      f"Cabem {TPM // _tk} por minuto.")
print()
print("O paralelismo deu 1.00? Era o esperado: nesta pergunta ha uma CADEIA")
print("DE DEPENDENCIA. Ele nao pode medir a camara antes de saber qual e.")

DESENHO FINO · um lote · com rastro
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 3 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-88"})
    -> L-88: meio de cultura; guardado na camara CF-01; faixa permitida de 2 a 8 C; validade 2026-09-15
--- volta 4 ----------------------------------------------
  temperatura_camara({"camara":"CF-01"})
    -> CF-01: 4.2 graus Celsius neste momento
--- volta 5 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 6 ----

In [14]:
# ESPERE UM MINUTO. A calibracao ja consumiu ~47% do seu TPM.
import time; time.sleep(60)

base_fina = medir("fina", "P2")

  exec 1: 6 voltas | 5 chamadas (1.00 por volta) | evidencia 3/3 | cobriu os lotes | 4066+1483 tokens

  [fina · P2] modelo openai/gpt-oss-20b, n=1
  voltas 6.0 | chamadas 5.0 | PARALELISMO 1.00
  evidencia 3.00/3 | cobriu 1/1 | US$ 0.000750 por execucao
  tokens 5549 por execucao (69% do TPM)


In [15]:
# ESPERE UM MINUTO. A medicao anterior usou ~94% do TPM.
import time; time.sleep(60)

print("=" * 66)
print("DESENHO GROSSO · um lote · com rastro")
print("=" * 66)
_demo_g = agente(PERGUNTA_2, desenho="grossa", verboso=True)
print()
print(f"voltas {_demo_g['voltas']} | chamadas {_demo_g['chamadas']} | "
      f"evidencia {avaliar_evidencia(_demo_g['resposta'])}/3")
print()
print("Uma chamada. E o rastro NAO MOSTRA que tres consultas aconteceram")
print("por dentro — elas estao no Python, e o rastro do agente nao as ve.")

DESENHO GROSSO · um lote · com rastro
--- volta 1 ----------------------------------------------
  avaliar_lote({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
CF-02: 9.8 graus Celsius neste momento
CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 2 ----------------------------------------------
  avaliar_lote({"lote":"L-88"})
    -> L-88: meio de cultura; guardado na camara CF-01; faixa permitida de 2 a 8 C; validade 2026-09-15
CF-01: 4.2 graus Celsius neste momento
CF-01: 0 excursoes nas ultimas 24h
--- volta 3 ----------------------------------------------
RESPOSTA FINAL: **Avaliação dos lotes refrigerados**

| Lote | Temperatura atual (°C) | Faixa permitida (°C) | Tempo fora da faixa |
|------|------------------------|----------------------|---------------------|
| L‑77 | 9,8 °C | 2 – 8 °C | 3 h (a câmara CF‑02 está acima de 8 

In [16]:
import time; time.sleep(60)

base_grossa = medir("grossa", "P1")

  exec 1: 2 voltas | 1 chamadas (1.00 por volta) | evidencia 3/3 | cobriu os lotes | 783+487 tokens
  exec 2: 2 voltas | 1 chamadas (1.00 por volta) | evidencia 3/3 | cobriu os lotes | 783+577 tokens

  [grossa · P1] modelo openai/gpt-oss-20b, n=2
  voltas 2.0 | chamadas 1.0 | PARALELISMO 1.00
  evidencia 3.00/3 | cobriu 2/2 | US$ 0.000218 por execucao
  tokens 1315 por execucao (16% do TPM)


In [19]:
# A comparacao que vai ao quadro.
def comparar_desenhos(*resultados):
    """Tabela lado a lado. Passe dois ou mais resultados de medir()."""
    linhas = [
        ("ferramentas", lambda r: f"{len(DESENHOS[r['desenho']]['tools'])}"),
        ("declaracao (car.)", lambda r: f"{r['declaracao_car']}"),
        ("voltas", lambda r: f"{r['voltas']:.1f}"),
        ("chamadas", lambda r: f"{r['chamadas']:.1f}"),
        ("PARALELISMO", lambda r: f"{r['paralelismo']:.2f}"),
        ("entrada", lambda r: f"{r['entrada']:.0f}"),
        ("saida", lambda r: f"{r['saida']:.0f}"),
        ("tokens", lambda r: f"{r['tokens']:.0f}"),
        ("EVIDENCIA", lambda r: f"{r['evidencia']:.2f}/3"),
        ("US$ por execucao", lambda r: f"{r['custo_exec']:.6f}"),
    ]
    cab = "".join(f"{r['desenho'] + ' · ' + r['pergunta']:>22}" for r in resultados)
    print(f"{'':>20}{cab}")
    print("-" * (20 + 22 * len(resultados)))
    for rot, f in linhas:
        print(f"{rot:>20}" + "".join(f"{f(r):>22}" for r in resultados))

    if len(resultados) == 2:
        a, b = resultados
        print()
        if b["custo_exec"] and a["custo_exec"]:
            razao = a["custo_exec"] / b["custo_exec"]
            print(f">> o {b['desenho']} custou {1 / razao:.2f}x o {a['desenho']} "
                  f"({'mais barato' if razao > 1 else 'mais caro'})")
        d_ev = b["evidencia"] - a["evidencia"]
        print(f">> evidencia: {d_ev:+.2f}  "
              f"({'PIOROU' if d_ev < -0.4 else 'nao piorou'})")
        print()
        print("Se o grosso ficou mais barato E a evidencia nao caiu, entao em")
        print("tudo o que voce sabe medir hoje o WORKFLOW venceu o AGENTE.")
        print("A pergunta do bloco D e: o que voce NAO esta medindo?")


comparar_desenhos(base_fina, base_grossa)

                                 fina · P2           grossa · P1
----------------------------------------------------------------
         ferramentas                     3                     1
   declaracao (car.)                  1469                   444
              voltas                   6.0                   2.0
            chamadas                   5.0                   1.0
         PARALELISMO                  1.00                  1.00
             entrada                  4066                   783
               saida                  1483                   532
              tokens                  5549                  1315
           EVIDENCIA                3.00/3                3.00/3
    US$ por execucao              0.000750              0.000218

>> o grossa custou 0.29x o fina (mais barato)
>> evidencia: +0.00  (nao piorou)

Se o grosso ficou mais barato E a evidencia nao caiu, entao em
tudo o que voce sabe medir hoje o WORKFLOW venceu o AGENTE.
A pergunta do b

In [20]:
import time

print("=" * 66)
print("PARTE 10 — CAMINHO ESTENDIDO")
print("=" * 66)

# ---------------------------------------------------------------------
# 1. O Quarto Desenho: avaliar_lote com flag opcional
# ---------------------------------------------------------------------
print("\n[1] O Quarto Desenho: ferramenta grossa parametrizável")


def avaliar_lote_flex(lote: str, com_historico: bool = True) -> str:
    """Devolve a ficha e temperatura atual de um LOTE, com histórico opcional."""
    ficha = especificacao_do_insumo(lote)
    m = PADRAO_CAMARA.search(ficha)
    if not m:
        return ficha
    camara = m.group(0)
    leitura = temperatura_camara(camara)
    if com_historico:
        historico = historico_excursoes(camara, "24")
        return f"{ficha}\n{leitura}\n{historico}"
    return f"{ficha}\n{leitura}"


TOOLS_FLEX = [
    _f(
        "avaliar_lote_flex",
        "Devolve a ficha e temperatura de um LOTE. Se com_historico=True, "
        "inclui também as excursões das últimas 24 horas.",
        _param(
            "lote",
            _D_LOTE,
            extras={
                "com_historico": {
                    "type": "boolean",
                    "description": "Se True, busca o histórico de excursões.",
                }
            },
        ),
    )
]

DESENHOS["flex"] = {
    "tools": TOOLS_FLEX,
    "registro": {"avaliar_lote_flex": avaliar_lote_flex},
}

# Teste rápido do 4º desenho
r_flex = agente(PERGUNTA_1, desenho="flex", verboso=True)
print(
    f"Flex -> Voltas: {r_flex['voltas']} | Paralelismo: {paralelismo(r_flex):.2f}"
)

# ---------------------------------------------------------------------
# 2. Forçar o Paralelismo via Instrução (P2 no desenho fino)
# ---------------------------------------------------------------------
print("\n" + "-" * 66)
print("[2] Forçar Paralelismo via Prompt")
time.sleep(60)

INSTRUCAO_PARALELA = (
    INSTRUCAO
    + "\nOTIMIZAÇÃO: Quando precisar obter dados independentes (como de múltiplos lotes), "
    + "solicite todas as ferramentas necessárias de uma única vez na mesma volta."
)

r_paralelo = agente(
    PERGUNTA_2, desenho="fina", instrucao=INSTRUCAO_PARALELA, verboso=True
)
print(f"Fina + Prompt Paralelo -> Voltas: {r_paralelo['voltas']} | "
      f"Chamadas: {r_paralelo['chamadas']} | "
      f"Paralelismo: {paralelismo(r_paralelo):.2f}")

# ---------------------------------------------------------------------
# 4. Impacto do reasoning_effort="low"
# ---------------------------------------------------------------------
print("\n" + "-" * 66)
print("[4] Reduzindo esforço de raciocínio (reasoning_effort='low')")
time.sleep(60)

try:
    r_low = agente(
        PERGUNTA_1, desenho="fina", reasoning_effort="low", verboso=True
    )
    print(
        f"Fina (low reasoning) -> Saída: {r_low['saida']} tokens | Custo: US$ {custo_usd(r_low['entrada'], r_low['saida']):.6f}"
    )
except Exception as e:
    print(f"Nota: Modelo atual pode não suportar reasoning_effort via API Groq: {e}")

# ---------------------------------------------------------------------
# 5. Ferramenta de Escrita no Mundo: resolver_lote
# ---------------------------------------------------------------------
print("\n" + "-" * 66)
print("[5] Ferramenta com efeito colateral (Escrita / Mutação)")

BANCO_STATUS = {"L-77": "PENDENTE", "L-88": "PENDENTE"}


def registrar_decisao(lote: str, decisao: str, motivo: str) -> str:
    """Registra no sistema a decisão final sobre o insumo (APROVADO ou DESCARTADO)."""
    if lote not in BANCO_STATUS:
        return f"ERRO: Lote {lote} inexistente."
    BANCO_STATUS[lote] = f"{decisao.upper()} - {motivo}"
    return f"Lote {lote} atualizado com sucesso para: {BANCO_STATUS[lote]}"


TOOLS_ESCRITA = TOOLS_GROSSA + [
    _f(
        "registrar_decisao",
        "Registra formalmente a aprovação ou descarte de um lote no banco.",
        _param(
            "lote",
            _D_LOTE,
            extras={
                "decisao": {
                    "type": "string",
                    "enum": ["APROVADO", "DESCARTADO"],
                    "description": "Decisão sobre o uso.",
                },
                "motivo": {
                    "type": "string",
                    "description": "Justificativa técnica.",
                },
            },
        ),
    )
]

DESENHOS["escrita"] = {
    "tools": TOOLS_ESCRITA,
    "registro": {
        "avaliar_lote": avaliar_lote,
        "registrar_decisao": registrar_decisao,
    },
}

time.sleep(60)
r_escrita = agente(
    "Avalie e registre no sistema a decisão para o lote L-77.",
    desenho="escrita",
    verboso=True,
)
print("Estado do banco após execução:", BANCO_STATUS)

# ---------------------------------------------------------------------
# 6. Avaliador de Acurácia Semântica (Diferenciação L-77 vs L-88)
# ---------------------------------------------------------------------
print("\n" + "-" * 66)
print("[6] Avaliador Semântico / Lógica de Negócio")


def avaliar_decisao_correta_p2(resposta: str) -> bool:
    """Valida se a regra de negócio central foi atendida: L-77 NÃO e L-88 SIM."""
    t = _normalizar(resposta)

    # Identifica se L-77 foi rejeitado
    l77_rejeitado = bool(
        re.search(r"l-?77[^\.\n]*?(?:nao|bloque|descart|improprio|rejeit)", t)
    )

    # Identifica se L-88 foi aprovado
    l88_aprovado = bool(
        re.search(r"l-?88[^\.\n]*?(?:pode|liberad|aprovad|normal|ok|conforme)", t)
    )

    return l77_rejeitado and l88_aprovado


print(f"Decisão P2 na execução fina anterior correta? {avaliar_decisao_correta_p2(r_paralelo['resposta'])}")

PARTE 10 — CAMINHO ESTENDIDO

[1] O Quarto Desenho: ferramenta grossa parametrizável
--- volta 1 ----------------------------------------------
  avaliar_lote_flex({"com_historico":true,"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
CF-02: 9.8 graus Celsius neste momento
CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 2 ----------------------------------------------
RESPOSTA FINAL: **Avaliação do lote L‑77**

| Item | Valor |
|------|-------|
| **Leitura atual** | 9,8 °C |
| **Faixa permitida** | 2 °C – 8 °C |
| **Tempo fora da faixa** | 3 h (desde –3 h) |

**Conclusão**  
O lote L‑77 está atualmente fora da faixa de temperatura permitida (excesso de 1,8 °C) e permanece nessa condição há 3 horas. Até que a câmara retorne à faixa de 2 °C – 8 °C, o lote não pode ser utilizado.  

*Observação:* houve uma excursão anterior a –19 h que